# 13 · Model C: deep kernel learning (a network inside a Gaussian process)

*Level: advanced. Guide Section 17. Code: `deep.DeepKernelGP`.*

**Idea.**
* The CNN finds features but needs data and has weaker error bars.
* The GP has honest error bars and needs little data, but it only uses the inputs it is given.

A deep kernel puts the network in front of the GP: the GP works on the network's 16-number embedding (plus the settings),

$$k(i,i') = \sigma_f^2\,M\bigl((g_\omega(\text{curves}_i), x_i), (g_\omega(\text{curves}_{i'}), x_{i'})\bigr) + \sigma_n^2\mathbb 1[i=i'] .$$

There is no amplifier term here (`amp_term=False`): **no amplifier labels**.

In [ ]:
from tutorial_utils import *

from ml import deep
from ml.benchmark import THETA_CURVES, inputs_for
du, cu, _, _ = toy_scan(400, seed=7)
Xu, _ = inputs_for(du, cu, ("theta",), True, THETA_CURVES, log_frequency_spectra=True)
pre = deep.Pretrainer(epochs=20).fit(Xu["curves"], du[data.columns(du, "theta")].to_numpy(float))   # as in notebook 12

## 1. Frozen or fine-tuned encoder

* **Fine-tuned:** the encoder's weights and the GP's hyperparameters maximize the marginal likelihood **together**.
* **Frozen:** only the GP is fitted, on the fixed pre-trained embedding.

A flexible encoder can "over-fit" the marginal likelihood: it can map the training examples so that they look perfectly explained, which makes
the error bars too small. Watch the **coverage** column (target 0.68).

In [ ]:
dft, ct, Yt, _ = toy_scan(60, seed=2)
Xt, _ = inputs_for(dft, ct, ("theta", "ref"), True, THETA_CURVES, log_frequency_spectra=True)   # GP inputs: settings + own reference features
dtr, ctr, Ytr, _ = toy_scan(60, seed=1)
Xtr, _ = inputs_for(dtr, ctr, ("theta", "ref"), True, THETA_CURVES, log_frequency_spectra=True)
for label, finetune in (("deep kernel, encoder frozen", False), ("deep kernel, encoder fine-tuned", True)):
    dk = deep.DeepKernelGP(pre, finetune=finetune, amp_term=False)
    m, s = dk.fit({"curves": Xtr["curves"], "x": Xtr["scalars"]}, Ytr).predict({"curves": Xt["curves"], "x": Xt["scalars"]})
    _ = report(label, Yt, m, s)
m, s = models.GP().fit(Xtr["scalars"], Ytr).predict(Xt["scalars"]); _ = report("plain GP, settings + own reference features", Yt, m, s)

**Reading the numbers:** the fine-tuned version can have a small error but claims far too small error bars (its coverage is far below
0.68). That is the over-fitting of the marginal likelihood described above. The frozen version's error bars are honest.

## When to use it

* **Good:** with a frozen pre-trained encoder, it was the best network-based model for λ and the gain in our tests, with calibrated error bars. It is a
  candidate surrogate once traces are part of the inputs.
* **Bad:** fine-tuned with few packets (over-confident).

Next: **14 · Set models for the repeated trace sets (E).**